# EDA: аномалії у транзакціях мерчанта (Solidgate hackathon)

Датасет містить 1 000 000 транзакцій за 2025 рік без явної мітки `is_anomaly` — задача зводиться до **unsupervised anomaly detection**: аномалії потрібно виявляти через статистичні відхилення, невідповідності та штучні патерни, а не через навчання на мітках.

## План дослідження
1. Структура даних та типи
2. Цілісність (унікальність ключів, пропуски)
3. Базова статистика та розподіли категорій
4. Приведення часових колонок
5. Перевірка гіпотез про пропуски (crosstab)
6. Розслідування підозрілої групи `bank_id == 777`
7. Аналіз latency (processed_at - created_at) для всього датасету
8. Висновки та відкриті питання


## 1. Завантаження та структура даних

In [38]:
import pandas as pd

df = pd.read_csv('/home/veronika/Anomaly_Hunter_Solidgate/hackathon_int20h_dataset_test.csv')

In [39]:
print(df.shape)
print(df.head(5))


(1000000, 18)
            created_at  order_id         processed_at order_type  user_id  \
0  2025-07-01 09:21:23         1  2025-07-01 09:21:32      first   692925   
1  2025-09-01 01:15:47         2  2025-09-01 01:15:57  recurring   452913   
2  2025-06-24 23:38:35         3  2025-06-24 23:38:39      first   784680   
3  2025-04-23 04:42:13         4  2025-04-23 04:42:21      first   300037   
4  2025-03-14 20:15:32         5  2025-03-14 20:15:42      first   996803   

  ip_country currency  amount payment_method order_payment_type bin_country  \
0        DEU      EUR    4.60      googlepay                NaN         GBR   
1        CAN      CAD   54.80           card          recurring         CAN   
2        USA      USD    9.99           card                NaN         USA   
3        CAN      CAD    1.37           card                NaN         CAN   
4        DEU      EUR    0.92           card                NaN         GBR   

   bank_id     psp_id  has_refund  refunded_amou

In [40]:
print(df.head().info())
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   created_at          5 non-null      object 
 1   order_id            5 non-null      int64  
 2   processed_at        5 non-null      object 
 3   order_type          5 non-null      object 
 4   user_id             5 non-null      int64  
 5   ip_country          5 non-null      object 
 6   currency            5 non-null      object 
 7   amount              5 non-null      float64
 8   payment_method      5 non-null      object 
 9   order_payment_type  1 non-null      object 
 10  bin_country         5 non-null      object 
 11  bank_id             5 non-null      int64  
 12  psp_id              5 non-null      object 
 13  has_refund          5 non-null      bool   
 14  refunded_amount     5 non-null      float64
 15  is_secured          5 non-null      bool   
 16  status      

In [41]:
print(df.nunique())

created_at             983310
order_id              1000000
processed_at           983301
order_type                  2
user_id                603337
ip_country                  8
currency                    7
amount                    112
payment_method              3
order_payment_type          4
bin_country                 8
bank_id                    50
psp_id                      3
has_refund                  2
refunded_amount           201
is_secured                  2
status                      2
error_code                 13
dtype: int64


**Знахідка:** `created_at` і `processed_at` завантажились як `object`, хоча це часові мітки - потребують явної конвертації в `datetime` (зроблено нижче, розділ 4), інакше часовий аналіз неможливий.

## 2. Цілісність даних: унікальність ключа та пропуски

In [42]:
print(df['order_id'].duplicated().sum())

0


In [43]:
print(df.isna().sum())

created_at                 0
order_id                   0
processed_at               0
order_type                 0
user_id                    0
ip_country                 0
currency                   0
amount                     0
payment_method             0
order_payment_type    400526
bin_country                0
bank_id                    0
psp_id                     0
has_refund                 0
refunded_amount            0
is_secured                 0
status                     0
error_code            525114
dtype: int64


**Спостереження:** дублікатів `order_id` немає - ключ цілісний. Є два помітні блоки пропусків: `order_payment_type` (~400k) та `error_code` (~525k). Гіпотеза: обидва пов'язані не з поганою якістю даних, а зі структурною логікою (перевіряється crosstab-ами в розділі 5).

## 3. Базова статистика та розподіли категорій

In [44]:
print(df.describe())

             order_id         user_id          amount         bank_id  \
count  1000000.000000  1000000.000000  1000000.000000  1000000.000000   
mean    500000.500000   550083.804859      115.133693       25.474309   
std     288675.278933   259853.867901      390.972249       23.637364   
min          1.000000   100002.000000        0.780000        1.000000   
25%     250000.750000   324983.000000        9.990000       13.000000   
50%     500000.500000   550019.000000       20.100000       25.000000   
75%     750000.250000   775152.500000       50.000000       37.000000   
max    1000000.000000   999997.000000     8240.000000      777.000000   

       refunded_amount     error_code  
count   1000000.000000  474886.000000  
mean          2.339364       2.905473  
std          43.627719       0.497555  
min           0.000000       0.010000  
25%           0.000000       3.020000  
50%           0.000000       3.020000  
75%           0.000000       3.080000  
max        6190.000000

Далі - прохід `value_counts()` по всіх колонках ("полювання на рідкісні категорії"): шукаємо значення, які різко випадають з розподілу за частотою або форматом.

In [45]:
for col in df.columns:
    print(df[col].value_counts())
    print("\n") 

created_at
2025-03-17 16:23:43    4
2025-10-06 16:13:36    3
2025-08-07 20:02:12    3
2025-12-25 22:17:05    3
2025-07-04 10:30:32    3
                      ..
2025-03-09 15:57:53    1
2025-12-01 12:16:55    1
2025-12-26 08:35:56    1
2025-12-29 15:12:01    1
2025-12-01 11:20:19    1
Name: count, Length: 983310, dtype: int64


order_id
1          1
666658     1
666660     1
666661     1
666662     1
          ..
333338     1
333339     1
333340     1
333341     1
1000000    1
Name: count, Length: 1000000, dtype: int64


processed_at
2025-06-20 23:35:28    4
2025-11-23 15:41:42    4
2025-09-26 23:12:47    4
2025-02-07 02:41:06    4
2025-11-14 00:24:20    3
                      ..
2025-06-03 22:00:40    1
2025-11-12 08:01:12    1
2025-04-17 09:08:00    1
2025-01-22 19:21:51    1
2025-12-01 11:20:24    1
Name: count, Length: 983301, dtype: int64


order_type
recurring    599474
first        400526
Name: count, dtype: int64


user_id
164513    10
703576     9
923569     9
221556     9
67

**Ключова знахідка:** `bank_id == 777` - лише 635 транзакцій, і сам ID тризначний, тоді як решта `bank_id` - одно-двозначні числа. Це нетиповий формат ID, кандидат на штучно додану/позначену групу. Детальне розслідування - у розділі 6.

Інші колонки (`psp_id`, решта `bank_id`, `currency`, `error_code`) подібної аномалії за кількістю входжень не показали.

## 4. Приведення часових колонок до datetime

In [51]:
df['created_at'] = pd.to_datetime(df['created_at'])
df['processed_at'] = pd.to_datetime(df['processed_at'])

In [52]:
df.head()

,created_at,order_id,processed_at,order_type,user_id,ip_country,currency,amount,payment_method,order_payment_type,bin_country,bank_id,psp_id,has_refund,refunded_amount,is_secured,status,error_code
0,2025-07-01 09:21:23,1,2025-07-01 09:21:32,first,692925,DEU,EUR,4.60,googlepay,NaN,GBR,32,psp_alpha,False,0.0,False,fail,3.02
1,2025-09-01 01:15:47,2,2025-09-01 01:15:57,recurring,452913,CAN,CAD,54.80,card,recurring,CAN,1,psp_alpha,False,0.0,False,success,NaN
2,2025-06-24 23:38:35,3,2025-06-24 23:38:39,first,784680,USA,USD,9.99,card,NaN,USA,32,psp_alpha,False,0.0,False,fail,2.01
3,2025-04-23 04:42:13,4,2025-04-23 04:42:21,first,300037,CAN,CAD,1.37,card,NaN,CAN,31,psp_gamma,False,0.0,False,fail,3.04
4,2025-03-14 20:15:32,5,2025-03-14 20:15:42,first,996803,DEU,EUR,0.92,card,NaN,GBR,39,psp_beta,False,0.0,False,fail,2.12


## 5. Перевірка гіпотез про пропуски (crosstab)

In [53]:
pd.crosstab(df['order_payment_type'], df['order_type'], margins=True, normalize='index')

order_type,recurring
order_payment_type,
1-click,1.0
rebill,1.0
recurring,1.0
retry,1.0
All,1.0


**Підтверджено:** `order_payment_type` заповнений виключно для `order_type == recurring` (частка 1.0 по кожному значенню); для `first` - завжди NaN. Пропуск не є дефектом даних, а прямим наслідком типу замовлення.

In [54]:
pd.crosstab(df['error_code'], df['status'], margins=True, normalize='index')

status,fail
error_code,
0.01,1.0
2.01,1.0
2.03,1.0
2.12,1.0
3.01,1.0
3.02,1.0
3.04,1.0
3.05,1.0
3.08,1.0


**Підтверджено:** `error_code` заповнюється відповідно до `status == fail` - логічно, адже код помилки має сенс лише для невдалої транзакції.

## 6. Розслідування підозрілої групи `bank_id == 777`

Профілюємо підмножину (635 рядків) тими самими інструментами, що й увесь датасет, і порівнюємо з глобальними показниками, а не оцінюємо ізольовано.

In [55]:
check_777 = df[df['bank_id'] == 777] 
check_777

,created_at,order_id,processed_at,order_type,user_id,ip_country,currency,amount,payment_method,order_payment_type,bin_country,bank_id,psp_id,has_refund,refunded_amount,is_secured,status,error_code
999365,2025-12-01 13:16:39,999366,2025-12-01 13:16:44,recurring,769549,CAN,CAD,41.10,card,rebill,USA,777,psp_gamma,False,0.0,False,fail,4.09
999366,2025-12-01 00:30:43,999367,2025-12-01 00:30:48,first,888226,USA,USD,9.99,googlepay,NaN,USA,777,psp_alpha,False,0.0,False,fail,4.09
999367,2025-12-01 18:04:56,999368,2025-12-01 18:05:01,recurring,704040,USA,USD,30.00,card,recurring,CAN,777,psp_gamma,False,0.0,False,fail,4.09
999368,2025-12-01 04:24:47,999369,2025-12-01 04:24:52,recurring,567238,MEX,MXN,740.00,card,recurring,MEX,777,psp_alpha,False,0.0,False,fail,4.09
999369,2025-12-01 11:00:05,999370,2025-12-01 11:00:10,first,945225,USA,USD,9.99,applepay,NaN,USA,777,psp_beta,False,0.0,False,fail,4.09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,2025-12-01 21:27:19,999996,2025-12-01 21:27:24,recurring,956515,USA,USD,200.00,card,retry,USA,777,psp_alpha,False,0.0,False,fail,3.08
999996,2025-12-01 22:16:31,999997,2025-12-01 22:16:36,first,177169,CAN,CAD,1.37,card,NaN,CAN,777,psp_alpha,False,0.0,False,fail,4.09
999997,2025-12-01 01:52:32,999998,2025-12-01 01:52:37,first,740896,CAN,CAD,13.70,card,NaN,CAN,777,psp_beta,False,0.0,False,fail,4.09
999998,2025-12-01 17:38:42,999999,2025-12-01 17:38:47,recurring,246155,POL,PLN,160.80,card,recurring,DEU,777,psp_gamma,False,0.0,False,fail,4.09


In [56]:
print(check_777.nunique())

created_at            633
order_id              635
processed_at          633
order_type              2
user_id               635
ip_country              8
currency                7
amount                 88
payment_method          3
order_payment_type      4
bin_country             8
bank_id                 1
psp_id                  3
has_refund              1
refunded_amount         1
is_secured              2
status                  1
error_code              4
dtype: int64


In [57]:
print(check_777.describe())

                          created_at        order_id  \
count                            635      635.000000   
mean   2025-12-01 12:12:57.272441088   999683.000000   
min              2025-12-01 00:00:42   999366.000000   
25%       2025-12-01 06:00:53.500000   999524.500000   
50%              2025-12-01 12:06:46   999683.000000   
75%              2025-12-01 18:35:45   999841.500000   
max              2025-12-01 23:59:55  1000000.000000   
std                              NaN      183.452991   

                        processed_at        user_id       amount  bank_id  \
count                            635     635.000000   635.000000    635.0   
mean   2025-12-01 12:13:02.272441088  529282.642520   112.904504    777.0   
min              2025-12-01 00:00:47  101570.000000     0.780000    777.0   
25%       2025-12-01 06:00:58.500000  303089.500000     9.995000    777.0   
50%              2025-12-01 12:06:51  517291.000000    23.400000    777.0   
75%              2025-12-01 18:35

In [58]:
for col in check_777.columns:
    print(check_777[col].value_counts())
    print("\n") 

created_at
2025-12-01 09:46:33    2
2025-12-01 07:41:04    2
2025-12-01 13:16:39    1
2025-12-01 14:10:35    1
2025-12-01 11:30:18    1
                      ..
2025-12-01 08:07:56    1
2025-12-01 04:50:25    1
2025-12-01 20:16:26    1
2025-12-01 22:41:24    1
2025-12-01 11:20:19    1
Name: count, Length: 633, dtype: int64


order_id
999366     1
999792     1
999785     1
999786     1
999787     1
          ..
999578     1
999579     1
999580     1
999581     1
1000000    1
Name: count, Length: 635, dtype: int64


processed_at
2025-12-01 09:46:38    2
2025-12-01 07:41:09    2
2025-12-01 13:16:44    1
2025-12-01 14:10:40    1
2025-12-01 11:30:23    1
                      ..
2025-12-01 08:08:01    1
2025-12-01 04:50:30    1
2025-12-01 20:16:31    1
2025-12-01 22:41:29    1
2025-12-01 11:20:24    1
Name: count, Length: 633, dtype: int64


order_type
recurring    389
first        246
Name: count, dtype: int64


user_id
769549    1
552296    1
852552    1
488102    1
177941    1
         .

**Спростовані гіпотези** (збігаються з глобальним розподілом, отже не є ознакою аномальності саме цієї групи):
- `is_secured` rate = ~4% - так само, як і в усьому датасеті;
- частка "круглих" сум (20.0, 30.0, 5.0...) - глобально теж дуже поширена;
- географія (`ip_country`/`bin_country`, домінує USA) - приблизно відповідає загальним пропорціям.

**Підтверджені відмінності** (реальний "відбиток" групи):
- усі 635 транзакцій створені **в один календарний день** (години - розкидані, дата - фіксована);
- усі мають `status == fail`, `has_refund == False`, `refunded_amount == 0.0`;
- `order_id` цієї групи - **найбільші в датасеті** (група фізично в кінці файлу);
- формат `bank_id` (777, тризначний) не відповідає патерну решти банків.

Перевірка latency цієї групи - нижче.

In [62]:
duration = check_777['processed_at'] - check_777['created_at']
print(duration.nunique())
print(duration.describe())


1
count                635
mean     0 days 00:00:05
std      0 days 00:00:00
min      0 days 00:00:05
25%      0 days 00:00:05
50%      0 days 00:00:05
75%      0 days 00:00:05
max      0 days 00:00:05
dtype: object


**Сильний доказ штучності:** latency (processed_at - created_at) для всіх 635 рядків рівно **5 секунд, без жодної варіації**. Реальна мережева/банківська обробка так поводитись не може - це статистична сигнатура синтетично згенерованої/доданої групи.

## 7. Latency для всього датасету

Мета: перевірити, чи `bank_id == 777` - єдина штучно додана група, чи є ще приховані кластери з власною константною затримкою (в описовій статистиці по всьому датасету 635 рядків із 1M статистично непомітні).

In [63]:
duration = df['processed_at'] - df['created_at']
print(duration.nunique())
print(duration.describe())

2849
count                      1000000
mean        0 days 00:00:35.362397
std      0 days 00:06:44.623737607
min                0 days 00:00:02
25%                0 days 00:00:04
50%                0 days 00:00:06
75%                0 days 00:00:08
max                0 days 01:59:59
dtype: object


In [64]:
for id in df['bank_id'].unique():
    i = df[df['bank_id'] == id] 
    duration = i['processed_at'] - i['created_at']
    print(duration.nunique())
    print(duration.describe())

123
count                        20257
mean     0 days 00:00:37.192920965
std      0 days 00:06:59.486361362
min                0 days 00:00:02
25%                0 days 00:00:04
50%                0 days 00:00:06
75%                0 days 00:00:08
max                0 days 01:59:57
dtype: object
124
count                        20687
mean     0 days 00:00:35.787209358
std      0 days 00:06:42.800718590
min                0 days 00:00:02
25%                0 days 00:00:04
50%                0 days 00:00:06
75%                0 days 00:00:08
max                0 days 01:59:30
dtype: object
109
count                        20708
mean     0 days 00:00:33.617925439
std      0 days 00:06:33.026824349
min                0 days 00:00:02
25%                0 days 00:00:04
50%                0 days 00:00:06
75%                0 days 00:00:08
max                0 days 01:59:50
dtype: object
111
count                        20373
mean     0 days 00:00:34.538801354
std      0 days 00:06:41.9717869

**Проміжний висновок:** розподіл latency по всьому датасету виглядає природним (без додаткових константних піків), окрім максимуму `0 days 01:59:59` - підозріло близько до рівно 2 годин, що натякає на можливий технічний timeout. Перевірка по решті значень `bank_id` не виявила інших груп зі стисненим/константним розподілом - окрім 777, усе плюс-мінус відповідає загальній картині.

## Відкриті питання на наступну сесію
- Чи є кластер транзакцій біля межі `01:59:59` (не поодинокий викид, а група), і як він пов'язаний зі `status`/`error_code` (гіпотеза: timeout -> fail)?
- `value_counts()` latency, округленої до секунд, по всьому датасету - пошук інших прихованих "константних" піків, замаскованих у `describe()`.
- Систематичний drill-down по інших ID-колонках (`psp_id`, `currency`) за тим самим методом, що виявив `bank_id == 777`.
- Перехід від виявлення однієї штучної групи до ширшого підходу: rule-based ознаки плюс unsupervised методи (Isolation Forest / LOF) для менш очевидних аномалій.